# Term Project — Task 1: Image Classification (Fruit & Style)

학번 202502204

**추론 전용 노트북.** 학습된 모델(EfficientNet-B0 + 2-head)을 Google Drive에서 받아 test 이미지의 fruit/style 라벨을 예측하고 `./release/202502204.test.task1.txt` 를 생성한다.

- 출력 형식: 줄마다 `img_id\tstyle_label\tfruit_label` (탭 구분)
- Fruit: 0 apple · 1 asian pear · 2 banana · 3 cherry · 4 grape · 5 pineapple
- Style: 0 pencil color · 1 oil painting · 2 water color

In [ ]:
# Colab 기본 torch/torchvision 사용. 추가 패키지만 버전 고정 설치.
!pip install -qqq gdown==5.2.0

In [ ]:
# ===== 설정 =====
STUDENT_ID = "202502204"

# Task1 학습 모델(task1.pt)의 Google Drive 파일 ID
MODEL_FILE_ID = "12bq6yp7NkHreXwQZz6zH73bNrwu5zZSG"

# 평가용 test 데이터 위치. TA가 test 이미지를 data/test/images/ 아래에 두거나,
# test.zip 의 Google Drive ID 를 아래에 지정하면 자동 다운로드/해제한다.
DATA_ROOT = "data/test"
TEST_ZIP_ID = ""  # 비우면 data/test/images 가 이미 존재해야 함

BATCH_SIZE = 64
IMG_SIZE = 224

In [ ]:
import os, zipfile, gdown

# --- test 데이터 준비 ---
def has_images(root):
    if not os.path.isdir(root):
        return False
    for _, _, fs in os.walk(root):
        if any(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in fs):
            return True
    return False

if not has_images(DATA_ROOT):
    if TEST_ZIP_ID:
        gdown.download(f"https://drive.google.com/uc?id={TEST_ZIP_ID}", "test.zip", quiet=False)
        os.makedirs("data", exist_ok=True)
        with zipfile.ZipFile("test.zip") as z:
            z.extractall("data")
    assert has_images(DATA_ROOT), (
        f"test 이미지를 찾을 수 없습니다. {DATA_ROOT}/images/ 아래에 이미지를 두거나 TEST_ZIP_ID 를 설정하세요.")
print("data ok:", DATA_ROOT)

In [ ]:
# --- 학습된 모델 다운로드 ---
MODEL_PATH = "task1.pt"
if not os.path.exists(MODEL_PATH):
    gdown.download(f"https://drive.google.com/uc?id={MODEL_FILE_ID}", MODEL_PATH, quiet=False)
print("model:", MODEL_PATH, os.path.getsize(MODEL_PATH), "bytes")

In [ ]:
# --- 모델 정의 (학습 코드와 동일 구조) ---
import torch, torch.nn as nn
from torchvision import models, transforms

NUM_FRUIT, NUM_STYLE = 6, 3

class DualHeadClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.efficientnet_b0(weights=None)
        in_feats = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.dropout = nn.Dropout(0.2)
        self.fruit_head = nn.Linear(in_feats, NUM_FRUIT)
        self.style_head = nn.Linear(in_feats, NUM_STYLE)
    def forward(self, x):
        f = self.dropout(self.backbone(x))
        return self.fruit_head(f), self.style_head(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(MODEL_PATH, map_location=device)
state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
IMG_SIZE = ckpt.get("img_size", IMG_SIZE) if isinstance(ckpt, dict) else IMG_SIZE
model = DualHeadClassifier().to(device)
model.load_state_dict(state)
model.eval()

tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
print("loaded on", device, "| img_size", IMG_SIZE)

In [ ]:
# --- test 이미지 수집 (재귀, id 숫자 정렬) ---
from glob import glob
EXTS = (".jpg", ".jpeg", ".png")
files = [p for p in glob(os.path.join(DATA_ROOT, "**", "*"), recursive=True)
         if p.lower().endswith(EXTS)]
def id_key(p):
    stem = os.path.splitext(os.path.basename(p))[0]
    return (0, int(stem)) if stem.isdigit() else (1, stem)
files = sorted(set(files), key=id_key)
print("test images:", len(files))

In [ ]:
# --- 추론 & 출력 ---
from PIL import Image
os.makedirs("release", exist_ok=True)
out_path = f"release/{STUDENT_ID}.test.task1.txt"

@torch.no_grad()
def predict_batch(paths):
    imgs = torch.stack([tf(Image.open(p).convert("RGB")) for p in paths]).to(device)
    fp, sp = model(imgs)
    return fp.argmax(1).cpu().tolist(), sp.argmax(1).cpu().tolist()

with open(out_path, "w") as f:
    for i in range(0, len(files), BATCH_SIZE):
        batch = files[i:i + BATCH_SIZE]
        fr, st = predict_batch(batch)
        for path, fruit_label, style_label in zip(batch, fr, st):
            img_id = os.path.basename(path)
            print(f"{img_id}\t{style_label}\t{fruit_label}", file=f)
print("wrote", out_path)

In [ ]:
# --- 출력 확인 ---
with open(out_path) as f:
    lines = f.read().splitlines()
print("lines:", len(lines))
print("head:")
print("\n".join(lines[:5]))